# Test Suite 01: Landing Zone Gate & Integrity Verification
**Target:** Landing raw files (`output/raw/dt=*/events.json`, `output/dims/*.json`, `manifest.json`)

**Tests Covered:**
- L1: Landing directory structure & non-empty file assertion
- L2: JSON schema-on-read parseability
- L3: SHA-256 manifest checksum validation
- L4: Zero PII leakage gate in raw landing files


In [ ]:
%run ../notebooks/00_config

import json
import os
import hashlib
import glob

print(f"Testing Landing Zone at: {LANDING_ROOT}")


### Test L1: Directory Structure & File Presence


In [ ]:
raw_dir = os.path.join(LANDING_ROOT, "raw")
dims_dir = os.path.join(LANDING_ROOT, "dims")

assert os.path.exists(raw_dir), f"FAIL: Raw directory not found: {raw_dir}"
assert os.path.exists(dims_dir), f"FAIL: Dimensions directory not found: {dims_dir}"

dt_dirs = [d for d in os.listdir(raw_dir) if d.startswith("dt=")]
assert len(dt_dirs) > 0, f"FAIL: No dt= partitions found in {raw_dir}"

print(f"[PASS] L1: Directory structure verified. Found {len(dt_dirs)} dt= partitions.")


### Test L2: JSON Schema-on-Read & Structural Parseability


In [ ]:
corrupt_count = 0
valid_count = 0

sample_partitions = dt_dirs[:10]  # Validate sample or all partitions
for dt in sample_partitions:
    ev_file = os.path.join(raw_dir, dt, "events.json")
    if os.path.exists(ev_file):
        try:
            with open(ev_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            assert isinstance(data, list), f"Expected list in {ev_file}"
            valid_count += len(data)
        except Exception as e:
            print(f"Corruption in {ev_file}: {e}")
            corrupt_count += 1

assert corrupt_count == 0, f"FAIL: Found {corrupt_count} unparseable partition files"
print(f"[PASS] L2: Successfully parsed {len(sample_partitions)} sample partitions ({valid_count:,} events).")


### Test L3: SHA-256 Manifest Checksum Verification


In [ ]:
manifest_file = os.path.join(LANDING_ROOT, "manifest.json")
if os.path.exists(manifest_file):
    with open(manifest_file, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    
    mismatches = 0
    for rel_path, expected_hash in manifest.get("files", {}).items():
        full_path = os.path.join(LANDING_ROOT, rel_path)
        if os.path.exists(full_path):
            with open(full_path, "rb") as f:
                actual_hash = hashlib.sha256(f.read()).hexdigest()
            if actual_hash != expected_hash:
                print(f"Checksum mismatch for {rel_path}!")
                mismatches += 1
    assert mismatches == 0, f"FAIL: {mismatches} file checksum mismatches"
    print(f"[PASS] L3: Manifest SHA-256 checksums verified with 0 mismatches.")
else:
    print("[SKIP] L3: No manifest.json found in landing root (optional for ad-hoc runs).")


### Test L4: Landing Zero-PII Leakage Assertion


In [ ]:
pii_violations = 0
for dt in dt_dirs[:5]:
    ev_file = os.path.join(raw_dir, dt, "events.json")
    if os.path.exists(ev_file):
        with open(ev_file, "r", encoding="utf-8") as f:
            sample_records = json.load(f)[:50]
        for r in sample_records:
            if "ip_addr" in r or "ip_addr_decrypted" in r or "email" in r:
                pii_violations += 1

assert pii_violations == 0, f"FAIL: Detected {pii_violations} PII fields in landing files!"
print(f"[PASS] L4: Zero-PII compliance verified. Sensitive fields dropped prior to landing.")
